In [26]:
# =========================
# Generate recommendations and rebuild memory_with_actions for all dates
# =========================
import json
import re
import os
from llm_client import llm_call

dates = ["2025-11-01", "2025-11-15", "2025-11-29"]

os.makedirs("Memory", exist_ok=True)
os.makedirs("Recommendations", exist_ok=True)

def safe_parse_llm_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        json_match = re.search(r"\{.*\}", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    print("⚠️ Failed to parse LLM output")
    return None

system_prompt = """
You are an academic support assistant.

You will receive one student's memory card.
Your task is to recommend the most appropriate next action.

Possible actions:
- monitor
- check_in_with_student
- praise_student

Rules:
- Use only the provided signals.
- If the student has no current signals, usually recommend monitor.
- If the student has positive signals, praise_student may be appropriate.
- If the student has concerning negative signals, check_in_with_student may be appropriate.
- If the situation is mild or unclear, monitor may be appropriate.
- Return JSON only.

Return fields:
student_id, first_name, last_name, recommended_action, priority, short_reason
"""

for date_str in dates:
    print(f"\n===== Processing date: {date_str} =====")

    # -------------------------
    # Load memory
    # -------------------------
    with open(f"Memory/student_memory_{date_str}.json", "r", encoding="utf-8") as f:
        student_memory = json.load(f)

    students = student_memory.copy()
    print("Students to process:", len(students))

    # -------------------------
    # Generate recommendations
    # -------------------------
    results = []

    for s in students:
        user_prompt = f"""
Student:
{json.dumps(s, indent=2, ensure_ascii=False)}

Return JSON:
{{
  "student_id": 1,
  "first_name": "...",
  "last_name": "...",
  "recommended_action": "monitor",
  "priority": "medium",
  "short_reason": "..."
}}
"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        raw = llm_call(messages, max_new_tokens=200)
        parsed = safe_parse_llm_json(raw)

        if parsed:
            results.append(parsed)

    # -------------------------
    # Save recommendations
    # -------------------------
    with open(f"Recommendations/student_action_recommendations_{date_str}.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print("Saved recommendations:", len(results))

    # -------------------------
    # Merge recommendations into memory
    # -------------------------
    actions_by_student_id = {}

    for action in results:
        student_id = action.get("student_id")
        if student_id is None:
            continue

        actions_by_student_id[int(student_id)] = {
            "recommended_action": action.get("recommended_action"),
            "priority": action.get("priority"),
            "reason": action.get("short_reason"),
            "date": date_str
        }

    student_memory_with_actions = []

    for student in student_memory:
        student_id = int(student["student_id"])
        student_copy = dict(student)

        if student_id in actions_by_student_id:
            student_copy["actions"] = [actions_by_student_id[student_id]]
        else:
            student_copy["actions"] = []

        student_memory_with_actions.append(student_copy)

    with open(f"Memory/student_memory_with_actions_{date_str}.json", "w", encoding="utf-8") as f:
        json.dump(student_memory_with_actions, f, ensure_ascii=False, indent=2)

    print("Saved merged memory_with_actions")


===== Processing date: 2025-11-01 =====
Students to process: 12
Saved recommendations: 12
Saved merged memory_with_actions

===== Processing date: 2025-11-15 =====
Students to process: 7
Saved recommendations: 7
Saved merged memory_with_actions

===== Processing date: 2025-11-29 =====
Students to process: 7
Saved recommendations: 7
Saved merged memory_with_actions


In [27]:
# =========================
# Universal comparison for multiple dates
# =========================
import json
import pandas as pd
import os

dates = ["2025-11-01", "2025-11-15", "2025-11-29"]  # добавляй сколько угодно

os.makedirs("Comparison", exist_ok=True)

def flatten(data, date):
    rows = []

    for s in data:
        actions = s.get("actions", [])

        action_name = None
        priority = None

        if len(actions) > 0:
            action_name = actions[0].get("recommended_action")
            priority = actions[0].get("priority")

        rows.append({
            "student_id": s["student_id"],
            "FirstName": s["first_name"],
            "LastName": s["last_name"],
            "signal_count": len(s.get("signals", [])),
            "action": action_name,
            "priority": priority,
            "date": date
        })

    return pd.DataFrame(rows)


def classify(r):
    old_present = not pd.isna(r["signal_count_old"])
    new_present = not pd.isna(r["signal_count_new"])

    old_count = 0 if pd.isna(r["signal_count_old"]) else int(r["signal_count_old"])
    new_count = 0 if pd.isna(r["signal_count_new"]) else int(r["signal_count_new"])

    if (not old_present) and new_present and new_count > 0:
        return "new_risk"

    if old_present and old_count > 0 and (not new_present or new_count == 0):
        return "resolved"

    if new_count > old_count:
        return "worsened"

    if new_count < old_count:
        return "improved"

    if r["action_old"] != r["action_new"]:
        return "action_changed"

    return "stable"


# =========================
# Loop over date pairs
# =========================
for i in range(len(dates) - 1):
    old_date = dates[i]
    new_date = dates[i + 1]

    print(f"\n===== Comparing {old_date} → {new_date} =====")

    # -------------------------
    # Load memory_with_actions
    # -------------------------
    with open(f"Memory/student_memory_with_actions_{old_date}.json", "r", encoding="utf-8") as f:
        old_data = json.load(f)

    with open(f"Memory/student_memory_with_actions_{new_date}.json", "r", encoding="utf-8") as f:
        new_data = json.load(f)

    old_df = flatten(old_data, old_date)
    new_df = flatten(new_data, new_date)

    df = old_df.merge(
        new_df,
        on="student_id",
        how="outer",
        suffixes=("_old", "_new")
    )

    df["FirstName"] = df["FirstName_old"].combine_first(df["FirstName_new"])
    df["LastName"] = df["LastName_old"].combine_first(df["LastName_new"])

    df["change_type"] = df.apply(classify, axis=1)

    out = df[
        [
            "student_id",
            "FirstName",
            "LastName",
            "change_type",
            "signal_count_old",
            "signal_count_new",
            "action_old",
            "action_new",
            "priority_old",
            "priority_new"
        ]
    ].copy()

    out = out.sort_values(
        ["change_type", "LastName", "FirstName"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    file_path = f"Comparison/comparison_{old_date}_vs_{new_date}.csv"
    out.to_csv(file_path, index=False)

    print(f"Saved: {file_path}")
    print(out.head(10))


===== Comparing 2025-11-01 → 2025-11-15 =====
Saved: Comparison/comparison_2025-11-01_vs_2025-11-15.csv
   student_id FirstName  LastName     change_type  signal_count_old  \
0           9       Dan     Cohen  action_changed               1.0   
1          10       Noa      Levy  action_changed               1.0   
2          38        Or  Turgeman        improved               2.0   
3          29     Rotem     Golan        new_risk               NaN   
4          21       Guy   Abramov        resolved               1.0   
5          26      Amit     Cohen        resolved               1.0   
6          14      Maya     David        resolved               1.0   
7          25       Adi     Levin        resolved               2.0   
8          23      Idan    Ohayon        resolved               1.0   
9          13      Omer    Peretz        resolved               1.0   

   signal_count_new             action_old             action_new  \
0               1.0                monitor  